## 模型推理（可能需要重启 Notebook）

In [22]:
model_dir = "models/whisper-large-v2-asr-int8-round1"

language = "Chinese (China)"
language_abbr = "zh-CN"
language_decode = "chinese"
task = "transcribe"


### 使用 `PeftModel` 加载 LoRA 微调后 Whisper 模型

使用 `PeftConfig` 加载 LoRA Adapter 配置参数，使用 `PeftModel` 加载微调后 Whisper 模型

In [23]:
from transformers import AutoModelForSpeechSeq2Seq, AutoTokenizer, AutoProcessor
from peft import PeftConfig, PeftModel

peft_config = PeftConfig.from_pretrained(model_dir)

base_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    peft_config.base_model_name_or_path, load_in_8bit=True, device_map="auto"
)

peft_model = PeftModel.from_pretrained(base_model, model_dir)

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


In [24]:
tokenizer = AutoTokenizer.from_pretrained(peft_config.base_model_name_or_path, language=language, task=task)
processor = AutoProcessor.from_pretrained(peft_config.base_model_name_or_path, language=language, task=task)
feature_extractor = processor.feature_extractor

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### 使用 Pipeline API 部署微调后 Whisper 实现中文语音识别任务

In [25]:
test_audio = "data/audio/test.mp3"

In [26]:
from transformers import AutomaticSpeechRecognitionPipeline

pipeline = AutomaticSpeechRecognitionPipeline(model=peft_model, tokenizer=tokenizer, feature_extractor=feature_extractor)

forced_decoder_ids = processor.get_decoder_prompt_ids(language=language_decode, task=task)

In [27]:
import torch

with torch.cuda.amp.autocast():
    text = pipeline(test_audio, max_new_tokens=255)["text"]

/home/overman/miniconda3/envs/peft/lib/python3.10/site-packages/transformers/models/whisper/generation_whisper.py:480: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
/home/overman/miniconda3/envs/peft/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:316: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [28]:
text

'小莲今天就先从那些能帮助我自己对房地产整个市场建立认知的一些内容聊起来。比如说房地产市场的特点呀、它在经济体里的特殊性呀、为什么先发动权、房价受什么决定、我们如何看到未来房价的走势等等。你是不是感觉高度深度都有了呢?废话少说,咱们走起事实。房地产这个市场它在整个经济体当中是非常特殊而且非常重要的。咱们先来看到房地产市场的几个特点,然后咱们逐渐深入。首先啊房子这个东西一个最显而易见的关键点是什么呢?就是谁都需要。你可别小看这些废话啊。这要用专业的接地语输语呢,就是它不像股票、债券这些东西,它是高需。而且房子啊它不光是消费品,它也是个投资品。投资品就是长期它有可能会保值或升值。比如说有一天女朋友想买个少量的包你一看我去这东西怎么这么贵那女朋友就理直气壮的说你别看它贵但它长期可能会升值哎说噎着你一下就没话说了买买买这个呢就是投资品的一大特点就是它的价值并不取决于它实际对你带来的那个效应而取决于你预期它未来在市场上的价值就是别人越花多少钱来买它这就会导致投资品啊就价格越涨大家越买越买的它就越涨你要不就把它当个消费品啊？'